In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain accuracy                    0.647000                    0.478600   
             precision                   0.568923                    0.383021   
             recall                      0.612932                    0.490621   
             f1                          0.588895                    0.428874   
             kappa                       0.280646                   -0.037218   
             MCC                         0.282161                   -0.038231   
outputsTest  accuracy                    0.638900                    0.484900   
             precision                   0.557514                    0.382622   
             recall                      0.604310                    0.480268   
             f1                          0.579159                    0.424496   
             kappa                       0.263924                   -0.030878   
             MCC                         0.265137                   -0.031884   
outputsAll   accuracy                    0.656000                    0.480600   
             precision                   0.578582                    0.384698   
             recall                      0.624810                    0.489498   
             f1                          0.599874                    0.429328   
             kappa                       0.298880                   -0.034161   
             MCC                         0.300307                   -0.035239   

                        situation-dependent_vs_explicit  \
outputsTrain accuracy                          0.525100   
             precision                         0.616404   
             recall                            0.476882   
             f1                                0.536869   
             kappa                             0.064991   
             MCC                               0.067591   
outputsTest  accuracy                          0.525400   
             precision                         0.616460   
             recall                            0.476590   
             f1                                0.536718   
             kappa                             0.066135   
             MCC                               0.068574   
outputsAll   accuracy                          0.527700   
             precision                         0.619501   
             recall                            0.475159   
             f1                                0.536831   
             kappa                             0.071993   
             MCC                               0.074865   

                        non-persuasive_vs_persuasive  \
outputsTrain accuracy                       0.536400   
             precision                      0.394457   
             recall                         0.593916   
             f1                             0.472666   
             kappa                          0.088579   
             MCC                            0.095226   
outputsTest  accuracy                       0.530600   
             precision                      0.393221   
             recall                         0.585905   
             f1                             0.469498   
             kappa                          0.077303   
             MCC                            0.082498   
outputsAll   accuracy                       0.533900   
             precision                      0.389044   
             recall                         0.587965   
             f1                             0.466896   
             kappa                          0.083042   
             MCC                            0.089381   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTrain accuracy                   0.630600                  0.450200  
             precision                  0.307207                  0.123932  
             recall                     0.558868                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.282535,-0.035118,0.070343,0.089035,0.177775,-0.158191
accuracy,0.647300,0.481367,0.526067,0.533633,0.630767,0.449433
f1,0.589310,0.427566,0.536806,0.469687,0.394895,0.176152
kappa,0.281150,-0.034086,0.067706,0.082974,0.162567,-0.125923
precision,0.568340,0.383447,0.617455,0.392240,0.308036,0.122814
recall,0.614018,0.486796,0.476210,0.589262,0.559665,0.317207


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain pearson                    0.347441                    0.004879   
             spearman                   0.330347                    0.000129   
             MSE                        1.305119                    1.990242   
             RMSE                       1.140145                    1.409607   
             MAE                        0.883926                    1.092427   
outputsTest  pearson                    0.339791                   -0.010086   
             spearman                   0.315331                   -0.016385   
             MSE                        1.320418                    2.020171   
             RMSE                       1.147143                    1.420256   
             MAE                        0.889654                    1.099369   
outputsAll   pearson                    0.357540                   -0.002111   
             spearman                   0.341197                   -0.006454   
             MSE                        1.284921                    2.004223   
             RMSE                       1.131110                    1.414551   
             MAE                        0.874824                    1.098066   

                       situation-dependent_vs_explicit  \
outputsTrain pearson                          0.127474   
             spearman                         0.181117   
             MSE                              1.745052   
             RMSE                             1.319230   
             MAE                              1.039806   
outputsTest  pearson                          0.114342   
             spearman                         0.178243   
             MSE                              1.771316   
             RMSE                             1.328852   
             MAE                              1.037767   
outputsAll   pearson                          0.137442   
             spearman                         0.202388   
             MSE                              1.725117   
             RMSE                             1.311183   
             MAE                              1.026566   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTrain pearson                       0.103129                  0.111043   
             spearman                      0.094046                  0.145520   
             MSE                           1.793743                  1.777913   
             RMSE                          1.337193                  1.330948   
             MAE                           1.021576                  0.936756   
outputsTest  pearson                       0.106025                  0.104173   
             spearman                      0.098053                  0.138715   
             MSE                           1.787950                  1.791654   
             RMSE                          1.334919                  1.336128   
             MAE                           1.024241                  0.941294   
outputsAll   pearson                       0.098462                  0.110159   
             spearman                      0.087163                  0.143030   
             MSE                           1.803076                  1.779681   
             RMSE                          1.341306                  1.331855   
             MAE                           1.027022                  0.935504   

                       compressed_vs_elaborated  
outputsTrain pearson                  -0.072351  
             spearman                 -0.173340  
             MSE                       2.144702  
             RMSE                      1.462787  
             MAE                       1.129430  
outputsTest  pearson                  -0.066774  
             spearman                 -0.172689  
             MSE                       2.133549  
             RMSE                      1.459257  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.882801,1.096621,1.034713,1.024279,0.937851,1.131407
MSE,1.303486,2.004879,1.747162,1.794923,1.783083,2.138485
RMSE,1.139466,1.414805,1.319755,1.337806,1.332977,1.460801
pearson,0.348257,-0.002439,0.126419,0.102538,0.108459,-0.069243
spearman,0.328959,-0.007570,0.187249,0.093087,0.142422,-0.168121
